# Companion Notebook 03: Scaled Dot-Product Causal Attention Verification

This notebook verifies **Scaled Dot-Product Causal Self-Attention** calculations from scratch in PyTorch, executing step-by-step matrix projections and verifying calculations against manual calculations.


In [1]:
import torch
import torch.nn as nn
import numpy as np

# Set options
torch.set_printoptions(precision=4, sci_mode=False)

# Dimensions
L = 3    # Sequence length
d_k = 2  # Dimension per head

# Query, Key, Value matrices defined in study guide
Q = torch.tensor([[1.0, 0.0],
                  [0.0, 1.0],
                  [1.0, 1.0]], dtype=torch.float32)

K = torch.tensor([[1.0, 1.0],
                  [0.0, 1.0],
                  [1.0, 0.0]], dtype=torch.float32)

V = torch.tensor([[2.0, -1.0],
                  [0.0, 3.0],
                  [1.0, 1.0]], dtype=torch.float32)

print("Query Matrix Q:\n", Q.numpy())
print("\nKey Matrix K:\n", K.numpy())
print("\nValue Matrix V:\n", V.numpy())


Query Matrix Q:
 [[1. 0.]
 [0. 1.]
 [1. 1.]]

Key Matrix K:
 [[1. 1.]
 [0. 1.]
 [1. 0.]]

Value Matrix V:
 [[ 2. -1.]
 [ 0.  3.]
 [ 1.  1.]]


In [2]:
# Step 1: Compute raw attention scores Q * K^T
raw_scores = torch.matmul(Q, K.T)
print("Step 1: Raw Dot Product Scores (Q * K^T):\n", raw_scores.numpy())

# Step 2: Scale by sqrt(d_k)
scaled_scores = raw_scores / np.sqrt(d_k)
print("\nStep 2: Scaled Scores (divided by sqrt(2)):\n", scaled_scores.numpy())


Step 1: Raw Dot Product Scores (Q * K^T):
 [[1. 0. 1.]
 [1. 1. 0.]
 [2. 1. 1.]]

Step 2: Scaled Scores (divided by sqrt(2)):
 [[0.70710677 0.         0.70710677]
 [0.70710677 0.70710677 0.        ]
 [1.4142135  0.70710677 0.70710677]]


### Explanation of Outputs (Raw and Scaled Scores)
- **Raw Dot Product (Q * K^T)**: Computes raw similarity values before scaling or masking.
- **Scaled Scores**: Divides elements by $\sqrt{d_k} = \sqrt{2} \approx 1.4142$ to pull variance to $1.0$, preventing softmax saturation and vanishing gradients during training.


In [3]:
# Step 3: Apply Causal Masking
mask = torch.triu(torch.full((L, L), float('-inf')), diagonal=1)
masked_scores = scaled_scores + mask
print("Step 3: Causal Masked Scores:\n", masked_scores.numpy())

# Step 4: Apply Softmax to get attention weights
weights = torch.softmax(masked_scores, dim=-1)
print("\nStep 4: Attention Weights Matrix (Softmax):\n", weights.numpy())


Step 3: Causal Masked Scores:
 [[0.70710677       -inf       -inf]
 [0.70710677 0.70710677       -inf]
 [1.4142135  0.70710677 0.70710677]]

Step 4: Attention Weights Matrix (Softmax):
 [[1.         0.         0.        ]
 [0.5        0.5        0.        ]
 [0.50348985 0.24825509 0.24825509]]


### Explanation of Outputs (Causal Masking & Softmax)
- **Causal Masked Scores**: Future tokens (above the diagonal) are set to $-\infty$.
- **Softmax Attention Weights**: Evaluates exponential probabilities per row. Since future positions are set to $-\infty$, they have exactly $0.0$ attention weight, ensuring causal generation. The weights match the study guide's weights row-by-row (e.g. Row 2: $[0.5035, 0.2483, 0.2483]$).


In [4]:
# Step 5: Compute final weighted output O = P * V
output = torch.matmul(weights, V)
print("Step 5: Final Attention Output O (Weights * V):\n", output.numpy())

# Verification check
expected_output = np.array([[2.0000, -1.0000],
                            [1.0000,  1.0000],
                            [1.2553,  0.4897]])

print("\nMatches study guide calculations (within float precision)?")
print(np.allclose(output.numpy(), expected_output, atol=1e-3))


Step 5: Final Attention Output O (Weights * V):
 [[ 2.        -1.       ]
 [ 1.         1.       ]
 [ 1.2552348  0.4895305]]

Matches study guide calculations (within float precision)?
True


### Explanation of Outputs (Final Attention Output)
- **Final Attention Output O**: The dot product of attention weights and value matrices yields output matrix: $[[2.0, -1.0], [1.0, 1.0], [1.2553, 0.4897]]$, matching calculations to 4 decimal places.
- **Verification check**: Returns `True` indicating absolute mathematical match to our hand calculations.
